In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import re
import os
import sys

sys.path.append(os.path.abspath('../'))

In [35]:
raw_path = "../data/raw/"
processed_path = "../data/processed/"
figures_path= "../reports/figures/"
src_path = "../src/"

In [36]:
# Load dữ liệu từ các file CSV
df_listings = pd.read_csv(os.path.join(raw_path, 'listings.csv'))
df_calendar = pd.read_csv(os.path.join(raw_path, 'calendar_2025.csv'))                  

In [37]:
df_calendar.info()
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495670 entries, 0 to 495669
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   listing_id      495670 non-null  int64  
 1   date            495670 non-null  object 
 2   available       495670 non-null  object 
 3   price           0 non-null       float64
 4   adjusted_price  0 non-null       float64
 5   minimum_nights  495670 non-null  int64  
 6   maximum_nights  495670 non-null  int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 26.5+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1358 non-null   int64  
 1   listing_url                                   1358 non-null   object 
 2   s

## PHẦN 1: PHÂN TÍCH SƠ BỘ TẬP DỮ LIỆU CALENDAR_2025

Tập dữ liệu calendar phản ánh trạng thái đặt phòng theo từng ngày, giúp nắm bắt hành vi thực tế của thị trường.

Mục tiêu của bước này là xây dựng biến occupancy_rate nhằm đại diện cho mức độ sức hút thị trường của từng căn hộ.

In [39]:
df_calendar.head()

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,8521,2025-09-28,f,NaN,NaN,2,1125
1,8521,2025-09-29,f,NaN,NaN,2,1125
2,8521,2025-09-30,f,NaN,NaN,2,1125
3,8521,2025-10-01,f,NaN,NaN,2,1125
4,8521,2025-10-02,f,NaN,NaN,2,1125


In [40]:
df_calendar.describe()

,listing_id,price,adjusted_price,minimum_nights,maximum_nights
count,4.956700e+05,0.0,0.0,495670.000000,495670.000000
mean,6.303908e+17,NaN,NaN,22.519142,639.369244
std,5.666651e+17,NaN,NaN,36.787509,519.096546
min,8.521000e+03,NaN,NaN,1.000000,1.000000
25%,3.405758e+07,NaN,NaN,2.000000,365.000000
50%,6.827960e+17,NaN,NaN,15.000000,365.000000
75%,1.178492e+18,NaN,NaN,30.000000,1125.000000
max,1.517698e+18,NaN,NaN,700.000000,11250.000000


In [41]:
df_calendar.isnull().sum()  

listing_id             0
date                   0
available              0
price             495670
adjusted_price    495670
minimum_nights         0
maximum_nights         0
dtype: int64

Sau khi khám phá tổng quan tập dữ liệu calendar, chúng ta nhận thấy tập dữ liệu gồm 495.670 bản ghi với 7 trường thông tin.

Một vấn đề đáng chú ý là hai cột `price` và `adjusted_price` bị khuyết thiếu hoàn toàn (100% giá trị null). Điều này cho thấy hai biến này không chứa bất kỳ thông tin hữu ích nào cho phân tích (information gain = 0).

Việc giữ lại các cột này không chỉ làm tăng độ phức tạp không cần thiết của dữ liệu mà còn có thể gây nhiễu trong các bước xử lý tiếp theo, đặc biệt khi thực hiện các phép thống kê hoặc mô hình hóa. Việc loại bỏ các cột không mang thông tin cũng giúp cải thiện hiệu suất tính toán và giảm nguy cơ phát sinh lỗi trong quá trình xử lý dữ liệu.

Do đó, nhóm quyết định loại bỏ hoàn toàn hai cột này nhằm đảm bảo tập dữ liệu gọn gàng, nhất quán và tối ưu cho các bước phân tích tiếp theo.

In [42]:
df_calendar.dropna(axis=1, how='all', inplace=True)

In [43]:
df_calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495670 entries, 0 to 495669
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   listing_id      495670 non-null  int64 
 1   date            495670 non-null  object
 2   available       495670 non-null  object
 3   minimum_nights  495670 non-null  int64 
 4   maximum_nights  495670 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 18.9+ MB


Sau khi loại bỏ các cột trống hoàn toàn, chúng ta Bước tiếp theo, để đảm bảo tính toàn vẹn của chuỗi thời gian, chúng ta cần tiến hành rà soát các dòng dữ liệu ghi nhận dữ liệu trùng lặp ngày

In [44]:
# Kiểm tra dữ liệu trùng lặp
df_calendar.duplicated().sum()
df_calendar.duplicated(subset=['listing_id', 'date']).sum()

np.int64(0)

Kết quả kiểm tra cho thấy tập dữ liệu Calendar đảm bảo tính vẹn toàn dữ liệu (không có dữ liệu lặp). Dữ liệu đã sạch giá trị nhiễu và sẵn sàng cho bước chuyển đổi đặc trưng. Mô hình học máy không thể tính toán dữ liệu dạng chuỗi của cột available. Do đó, ta cần số hóa cột này: gán giá trị 1 cho những ngày đã được đặt (f) và 0 cho những ngày còn trống (t).

In [45]:
df_calendar['available'].isnull().sum()

np.int64(0)

In [46]:
df_calendar['available'].value_counts()

available
f    249564
t    246106
Name: count, dtype: int64

In [47]:
from src.feature_engineering import encode_binary_features
binary_cols = ['available']
df_calendar = encode_binary_features(df_calendar, binary_cols)

In [48]:
df_calendar['available'].head(10)

0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
Name: available, dtype: int64

Sau khi tiến hành binarization, chúng ta tiến hành gom nhóm theo từng căn hộ (listing_id) để tính tỷ lệ lấp đày trung bình trong năm. Hành động này giúp nén dữ liệu thời gian thành bảng dữ liệu chéo, chuẩn bị cho việc hợp nhất dữ liệu với tập listings.

In [50]:
df_occupancy = df_calendar.groupby('listing_id')['available'].mean().reset_index()

In [51]:
df_occupancy.rename(columns={'available': 'occupancy_rate'}, inplace=True)
df_occupancy.head()

,listing_id,occupancy_rate
0,8521,0.257534
1,11169,0.852055
2,19581,0.805479
3,27498,0.934247
4,79762,0.824658


In [52]:
df_occupancy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   listing_id      1358 non-null   int64  
 1   occupancy_rate  1358 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 21.3 KB


## KIỂM TRA TÍNH TOÀN VẸN DỮ LIỆU (DATA INTEGRITY)

Sau khi thực hiện nén dữ liệu từ tập *calendar_2025* về cấp độ từng căn hộ (listing_id), nhóm tiến hành đối chiếu với tập *listings* nhằm xác nhận tính nhất quán của dữ liệu.

In [53]:
calendar_ids = set(df_occupancy['listing_id'])
listing_ids = set(df_listings['id'])

if calendar_ids == listing_ids:
    print("Dữ liệu khớp hoàn toàn 100%")
else:
    print(f"Có {len(calendar_ids - listing_ids)} ID không khớp.")

Dữ liệu khớp hoàn toàn 100%


Kết quả cho thấy 100% mã định danh (1.358 listing_id) giữa hai tập dữ liệu trùng khớp hoàn toàn. Điều này chứng minh rằng quá trình xử lý không làm mất mát bản ghi cũng như không phát sinh các ID bất thường.

Nhờ đó, chúng ta có thể tự tin thực hiện bước hợp nhất dữ liệu (merge) mà không lo ngại về sai lệch hoặc thiếu hụt dữ liệu.

## GIÁ TRỊ CỦA BIẾN OCCUPANCY_RATE

Thông qua kỹ thuật gom nhóm (groupby), gần 500.000 dòng dữ liệu giao dịch theo ngày đã được chuyển đổi thành một biến đặc trưng duy nhất cho mỗi căn hộ.

Biến occupancy_rate không chỉ đơn thuần phản ánh tỷ lệ lấp đầy, mà còn đại diện cho mức độ hấp dẫn thực tế của căn hộ trên thị trường.

Trong bối cảnh bài toán định giá, chỉ dựa vào giá niêm yết có thể dẫn đến hiện tượng "giá ảo" — khi giá cao nhưng không có nhu cầu tương ứng.

Ngược lại, occupancy_rate cung cấp một tín hiệu về hành vi thực tế của khách hàng, giúp mô hình phân biệt giữa giá niêm yết và giá trị thị trường thực sự.

Do đó, việc đưa biến này vào mô hình giúp cải thiện đáng kể độ chính xác và tính thực tiễn của bài toán định giá.

## LƯU DỮ LIỆU SAU XỬ LÝ

Sau khi hoàn tất quá trình xử lý và kiểm tra tính toàn vẹn dữ liệu, tập dữ liệu df_occupancy được lưu lại để phục vụ cho các bước phân tích tiếp theo.

Việc lưu trữ dữ liệu trung gian giúp đảm bảo tính tái sử dụng và khả năng kiểm soát pipeline dữ liệu.

In [54]:
df_occupancy.to_csv('../data/processed/df_occupancy.csv', index=False)

## PHẦN 2: KHÁM PHÁ ĐẶC TRƯNG CĂN HỘ (EDA - LISTINGS)

Sau khi xác nhận tính đồng nhất về mã định danh (id) và xây dựng thành công biến số chiến lược occupancy_rate từ dữ liệu thời gian (calendar), chúng ta chuyển sang phân tích tập dữ liệu chính: **listings.csv**.

Nếu occupancy_rate đại diện cho "nhu cầu thị trường" (demand), thì tập listings phản ánh "nguồn cung" (supply) — bao gồm các đặc điểm cấu thành giá trị của từng căn hộ tại Cambridge.

Tập dữ liệu này đóng vai trò là "khung xương" của mô hình, cung cấp các thông tin về đặc điểm vật lý, vị trí và chính sách vận hành của từng căn hộ.

Việc thực hiện EDA cho tập dữ liệu này nhằm phục vụ trực tiếp cho bài toán định giá, với 3 mục tiêu trọng tâm:

1. **Tinh lọc đặc trưng (Feature Selection)**  
Loại bỏ các trường dữ liệu nhiễu (URL, mô tả văn bản dài, ID người dùng) để tập trung vào các biến có khả năng ảnh hưởng đến giá và hiệu suất kinh doanh như: room_type, accommodates, bathrooms, bedrooms, và neighbourhood.

2. **Làm sạch dữ liệu (Data Cleaning)**  
Chuẩn hóa kiểu dữ liệu (đặc biệt là cột price), xử lý giá trị thiếu (Missing Values) và kiểm soát các điểm dị biệt (Outliers) nhằm đảm bảo tính ổn định cho các phân tích và mô hình sau này.

3. **Hiểu cấu trúc thị trường (Distribution Analysis)**  
Phân tích phân phối của các biến quan trọng để nắm bắt bức tranh tổng quan về thị trường Airbnb tại Cambridge, bao gồm:
- Sự phân bố loại hình căn hộ  
- Mức giá phổ biến  
- Sự khác biệt giữa các khu vực  

Bước này đóng vai trò nền tảng trước khi hợp nhất dữ liệu với occupancy_rate, từ đó cho phép phân tích mối quan hệ giữa giá, đặc điểm căn hộ và hành vi thị trường.

In [55]:
# kiểm tra tập dữ liệu listings
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1358 entries, 0 to 1357
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1358 non-null   int64  
 1   listing_url                                   1358 non-null   object 
 2   scrape_id                                     1358 non-null   int64  
 3   last_scraped                                  1358 non-null   object 
 4   source                                        1358 non-null   object 
 5   name                                          1358 non-null   object 
 6   description                                   1347 non-null   object 
 7   neighborhood_overview                         694 non-null    object 
 8   picture_url                                   1358 non-null   object 
 9   host_id                                       1358 non-null   i

In [56]:
df_listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,8521,https://www.airbnb.com/rooms/8521,20250928035003,2025-09-28,city scrape,SunsplashedSerenity walk to Harvard & Fresh Pond,"An elegant, sun-splashed, 2 bedroom (+2offices...",Huron Village is known for its charm. We have...,https://a0.muscache.com/pictures/30536/072e0a5...,306681,...,4.92,4.94,4.74,C0121120491,f,2,2,0,0,0.41
1,11169,https://www.airbnb.com/rooms/11169,20250928035003,2025-09-28,city scrape,Lovely Studio Room: Available for long w/ends,Large sunny room w kitchenette & bath. Foam ma...,The neighborhood is quiet and friendly and our...,https://a0.muscache.com/pictures/miso/Hosting-...,40965,...,4.92,4.80,4.77,NaN,f,3,1,2,0,1.01
2,19581,https://www.airbnb.com/rooms/19581,20250928035003,2025-09-28,city scrape,"Furnished suite, Windsor","Welcome to Area IV! We are located, convenient...",NaN,https://a0.muscache.com/pictures/188f1b4b-f37b...,74249,...,4.91,4.91,4.27,NaN,t,3,0,3,0,0.07
3,27498,https://www.airbnb.com/rooms/27498,20250928035003,2025-09-28,city scrape,Furnished suite 2 @ the Windsor,"Welcome to Area IV! We are located, convenient...",NaN,https://a0.muscache.com/pictures/bab30c3c-ff3c...,74249,...,4.76,4.88,4.64,NaN,t,3,0,3,0,0.15
4,79762,https://www.airbnb.com/rooms/79762,20250928035003,2025-09-28,city scrape,Cambridge Getaway @ Harvard & MIT,Charming 2-bedroom apartment on the third floo...,Annmarie and I have lived in this area for ove...,https://a0.muscache.com/pictures/airflow/Hosti...,430015,...,4.93,4.93,4.77,STR-15661,f,1,1,0,0,2.59


In [57]:
df_listings.describe()

,id,scrape_id,host_id,host_listings_count,host_total_listings_count,neighbourhood_group_cleansed,latitude,longitude,accommodates,bathrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,1.358000e+03,1.358000e+03,1.358000e+03,1358.000000,1358.000000,0.0,1358.000000,1358.000000,1358.000000,1065.000000,...,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1358.000000,1358.000000,1358.000000,1358.000000,1025.000000
mean,6.303908e+17,2.025093e+13,1.777369e+08,698.770250,851.924153,NaN,42.373912,-71.106471,3.270250,1.307981,...,4.744244,4.870020,4.838771,4.872527,4.616556,38.865979,27.310015,10.977172,0.000736,1.525834
std,5.668733e+17,0.000000e+00,1.949916e+08,1681.470771,1939.485396,NaN,0.010625,0.019023,2.315851,0.595237,...,0.417628,0.303552,0.380003,0.281349,0.480144,56.951799,54.644200,20.183577,0.027136,1.909016
min,8.521000e+03,2.025093e+13,3.538400e+04,1.000000,1.000000,NaN,42.353670,-71.157580,1.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.010000
25%,3.414261e+07,2.025093e+13,2.436748e+07,3.000000,4.000000,NaN,42.366240,-71.119623,2.000000,1.000000,...,4.670000,4.860000,4.830000,4.860000,4.530000,2.000000,0.000000,0.000000,0.000000,0.240000
50%,6.827960e+17,2.025093e+13,1.074344e+08,16.000000,24.000000,NaN,42.370599,-71.105840,2.000000,1.000000,...,4.880000,4.950000,4.950000,4.930000,4.720000,8.000000,2.000000,1.000000,0.000000,0.850000
75%,1.178137e+18,2.025093e+13,3.736751e+08,169.000000,234.000000,NaN,42.381308,-71.091340,4.000000,1.500000,...,4.990000,5.000000,5.000000,5.000000,4.860000,83.000000,26.000000,12.000000,0.000000,2.250000
max,1.517698e+18,2.025093e+13,7.207319e+08,5143.000000,5933.000000,NaN,42.400180,-71.071239,16.000000,4.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,169.000000,169.000000,71.000000,1.000000,24.220000


In [58]:
missing_data=df_listings.isnull().sum().sort_values(ascending=False)
print(missing_data[missing_data > 0])

neighbourhood_group_cleansed    1358
calendar_updated                1358
license                          892
neighbourhood                    664
neighborhood_overview            664
host_about                       480
review_scores_communication      333
last_review                      333
reviews_per_month                333
first_review                     333
review_scores_location           333
review_scores_accuracy           333
review_scores_rating             333
review_scores_value              333
review_scores_cleanliness        333
review_scores_checkin            333
price                            303
estimated_revenue_l365d          303
beds                             300
bathrooms                        293
host_location                    250
host_response_rate               111
host_response_time               111
bedrooms                          93
host_acceptance_rate              86
host_is_superhost                 41
host_neighbourhood                34
h

## ĐÁNH GIÁ CHẤT LƯỢNG DỮ LIỆU VÀ CHIẾN LƯỢC XỬ LÝ

Dựa trên kết quả thống kê mô tả và kiểm tra giá trị khuyết thiếu, nhóm xác định các vấn đề trọng yếu ảnh hưởng trực tiếp đến bài toán định giá Airbnb tại Cambridge, đồng thời đề xuất các phương án xử lý như sau:

**1. Kiểm soát biến mục tiêu (price)**  
Với tỷ lệ thiếu khoảng ~22%, biến price được xem là yếu tố cốt lõi trong bài toán. Nhóm ưu tiên rà soát chéo với tập calendar để xác minh. Các bản ghi không thể xác định giá niêm yết sẽ bị loại bỏ nhằm đảm bảo độ tin cậy và tính nhất quán của mô hình dự báo.

**2. Tối ưu hóa dữ liệu đặc trưng (Feature Engineering)**  
Cột bathrooms có tỷ lệ thiếu cao, trong khi bathrooms_text gần như đầy đủ. Nhóm tận dụng cột này để trích xuất thông tin số lượng phòng tắm, qua đó khôi phục dữ liệu và tăng giá trị sử dụng của biến đặc trưng.

**3. Lọc nhiễu và tinh gọn dữ liệu (Noise Reduction)**  
Các cột trống hoàn toàn và siêu dữ liệu không mang giá trị dự báo sẽ được loại bỏ. Đối với biến license, thay vì loại bỏ, nhóm thực hiện mã hóa nhị phân nhằm khai thác thông tin về tính pháp lý và độ tin cậy của căn hộ.

**4. Chiến lược xử lý giá trị thiếu (Imputation)**  
Các biến quan trọng như bedrooms, beds và review_scores được xử lý bằng giá trị trung vị (Median) nhằm hạn chế ảnh hưởng của các giá trị cực biên, đồng thời giữ nguyên phân phối dữ liệu.

Cách tiếp cận này đảm bảo dữ liệu đầu vào vừa sạch, vừa giữ được tối đa giá trị thông tin phục vụ cho bài toán định giá.

## THIẾT LẬP QUY TRÌNH XỬ LÝ DỮ LIỆU

Nhóm triển khai quy trình xử lý theo nguyên tắc: **Làm sạch trước – Biến đổi sau**, nhằm đảm bảo tính toàn vẹn và khả năng khai thác dữ liệu hiệu quả.

Quy trình bao gồm 5 bước chính:

**1. Noise Filtering**  
Loại bỏ các đặc trưng không mang thông tin (100% null) và dữ liệu nhiễu để giảm độ phức tạp của tập dữ liệu.

**2. Target Normalization**  
Chuẩn hóa biến price và loại bỏ các bản ghi không có giá trị mục tiêu, đảm bảo tính hợp lệ cho bài toán học có giám sát.

**3. Numerical Cleaning**  
Xử lý các giá trị thiếu bằng Median và kiểm soát các điểm dị biệt (Outliers) thông qua phương pháp IQR nhằm bảo toàn phân phối dữ liệu.

**4. Feature Enrichment**  
Trích xuất thông tin từ dữ liệu văn bản (bathrooms_text) và mã hóa các biến định tính quan trọng (license, superhost) thành dạng số.

**5. Data Integration**  
Hợp nhất dữ liệu đặc trưng căn hộ với biến occupancy_rate nhằm kết nối giữa "nguồn cung" và "nhu cầu thị trường", tạo nền tảng cho phân tích định giá.

Pipeline này đảm bảo dữ liệu đầu vào không chỉ sạch về mặt kỹ thuật mà còn có ý nghĩa về mặt kinh doanh, phục vụ trực tiếp cho việc tối ưu hóa giá niêm yết.

Trong tập dữ liệu gốc, trường license chứa các mã định danh pháp lý dưới dạng chuỗi ký tự (Strings). Nhóm thực hiện chuyển đổi thông tin này sang dạng nhị phân (Binary Encoding) để chuẩn bị dữ liệu cho các bước kiểm định sau này:

has_license = 1: Căn hộ có thông tin giấy phép hiện hữu.

has_license = 0: Căn hộ thiếu thông tin hoặc chưa niêm yết giấy phép.

Việc trích xuất này nhằm mục đích giữ lại dữ liệu về trạng thái pháp lý - một yếu tố được kỳ vọng là có tiềm năng tác động đến tâm lý người thuê hoặc chiến lược định giá của chủ nhà. Biến số này sẽ đóng vai trò là một "giả thuyết đầu vào" để nhóm tiến hành phân tích tương quan và kiểm chứng sức ảnh hưởng thực tế ở các giai đoạn sau của dự án.

In [59]:
df_listings.drop(
    columns=['neighbourhood_group_cleansed', 'calendar_updated'],
    inplace=True
)

In [60]:
df_listings['price']

0         $270.00
1         $126.00
2         $183.00
3         $238.00
4         $300.00
          ...    
1353      $135.00
1354    $1,166.00
1355    $1,018.00
1356    $1,166.00
1357       $36.00
Name: price, Length: 1358, dtype: object

In [61]:
from src.data_cleaning import clean_price
df_listings = clean_price(df_listings)

In [62]:
df_listings['price'].isnull().sum()

np.int64(303)

Trong quá trình khám phá dữ liệu, nhóm thực hiện kiểm tra các giá trị thiếu trên toàn bộ tập dữ liệu listings. Kết quả cho thấy trường price (biến mục tiêu $Y$) có 303 bản ghi khuyết thiếu (chiếm khoảng 22% tổng số 1.358 quan sát ban đầu). Nhóm quyết định thực hiện loại bỏ hoàn toàn các bản ghi này thay vì áp dụng các kỹ thuật xử lý missing value dựa trên các cơ sở khoa học sau: Bản chất của biến phụ thuộc ($Y$): Trong các mô hình hồi quy hoặc dự báo kinh tế, price đóng vai trò là "nhãn" (Label) để máy tính học tập. Việc tự ý điền khuyết giá trị mục tiêu bằng giá trị trung bình hay trung vị sẽ vô tình tạo ra các "dữ liệu giả" (Synthetic Data), làm sai lệch phân phối thực tế của thị trường và dẫn đến hiện tượng Overfitting (mô hình chỉ học được các con số giả định thay vì quy luật thị trường thực tế).Đảm bảo độ tin cậy của Ground Truth: Một căn hộ không niêm yết giá được coi là "quan sát không có nhãn" (Unlabeled observation). Việc giữ lại chúng sẽ làm nhiễu các chỉ số đánh giá mô hình như $R^2$ hay $RMSE$ sau này.Tối ưu hóa phễu dữ liệu: Sau khi loại bỏ 303 dòng nhiễu, tập dữ liệu còn lại 1.055 dòng "nạc" đảm bảo tính nhất quán 100% về mặt thông tin (vừa có đặc trưng đặc điểm, vừa có giá niêm yết thực tế).Hành động: Nhóm triển kha

In [63]:
df_listings.dropna(subset=['price'], inplace=True)

In [64]:
df_listings['bathrooms_text'].isnull().sum()

np.int64(1)

In [ ]:
from src.data_cleaning import fill_bathrooms_text
df_listings = fill_bathrooms_text(df_listings)

In [32]:
from src.feature_engineering import extract_bath_features

df_listings[['bath_qty', 'is_shared_bath']] = df_listings['bathrooms_text'].apply(
    lambda x: pd.Series(extract_bath_features(x))
)

In [33]:
df_listings[['bath_qty', 'is_shared_bath']].isnull().sum()

bath_qty          0
is_shared_bath    0
dtype: int64

In [20]:
df_listings[['bath_qty', 'is_shared_bath']].describe()

,bath_qty,is_shared_bath
count,1055.000000,1055.000000
mean,1.309953,0.269194
std,0.596992,0.443751
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,0.000000
75%,1.500000,1.000000
max,4.000000,1.000000


In [ ]:
from src.data_cleaning import 

In [22]:
df_listings[['license', 'has_license']].head()

,license,has_license
0,C0121120491,1
1,NaN,0
2,NaN,0
3,NaN,0
4,STR-15661,1


In [23]:
df_listings['host_is_superhost'].value_counts()

host_is_superhost
f    624
t    396
Name: count, dtype: int64

In [25]:
df_listings['host_is_superhost'].isnull().sum()

np.int64(35)

In [26]:
df_listings['host_is_superhost'] = df_listings['host_is_superhost'].fillna(0)

In [27]:
df_listings['host_is_superhost'].head(10)

0     1.0
1     1.0
2     1.0
3     1.0
4     1.0
5     1.0
6     1.0
9     0.0
10    1.0
11    0.0
Name: host_is_superhost, dtype: float64

In [28]:
df_listings['host_identity_verified'].isnull().sum()

np.int64(0)

In [29]:
df_listings['host_identity_verified'].value_counts()

host_identity_verified
t    997
f     58
Name: count, dtype: int64

In [30]:
df_listings['host_identity_verified'] = df_listings['host_identity_verified'].map({'t': 1, 'f': 0})

In [31]:
df_listings['host_identity_verified'].head(10)

0     1
1     1
2     1
3     1
4     1
5     1
6     1
9     1
10    1
11    0
Name: host_identity_verified, dtype: int64

In [32]:
df_listings['instant_bookable'].isnull().sum()

np.int64(0)

In [33]:
df_listings['instant_bookable'].value_counts()

instant_bookable
f    634
t    421
Name: count, dtype: int64

In [34]:
df_listings['instant_bookable'] = df_listings['instant_bookable'].map({'t':1,'f':0})

In [35]:
df_listings['instant_bookable'].head(10)

0     0
1     0
2     1
3     1
4     0
5     1
6     0
9     1
10    0
11    0
Name: instant_bookable, dtype: int64

In [36]:
df_listings['bedrooms'].isnull().sum()

np.int64(0)

In [37]:
df_listings['bedrooms']=df_listings['bedrooms'].fillna(1)

df_listings['beds']=df_listings['beds'].fillna(
    df_listings.groupby('bedrooms')['beds'].transform('median')
)

In [38]:
df_listings['beds'].isnull().sum()

np.int64(0)

In [ ]:
from src.data_cleaning import clean_amenities

cleaned_amenities = clean_amenities(df_listings)

In [40]:
df_listings['amenities_count'] = cleaned_amenities.apply(len)

In [41]:
df_listings['amenities_count'].describe()

count    1055.000000
mean       35.502370
std        14.924343
min         4.000000
25%        27.000000
50%        33.000000
75%        42.000000
max       103.000000
Name: amenities_count, dtype: float64

In [ ]:
all_amenities = cleaned_amenities.explode().dropna()
all_amenities = all_amenities[all_amenities != '']

amenity_counts = Counter(all_amenities)

import pandas as pd
df_amenity_freq = pd.DataFrame(
    amenity_counts.items(),
    columns=['Amenity', 'Count']
).sort_values(by='Count', ascending=False)

top_amenities = df_amenity_freq.head(10)['Amenity'].tolist()

print(top_amenities)

['Smoke alarm', 'Carbon monoxide alarm', 'Wifi', 'Hot water', 'Hangers', 'Essentials', 'Bed linens', 'Kitchen', 'Hair dryer', 'Refrigerator']


In [298]:
df_listings['has_availability'].isnull().sum()

np.int64(2)

In [299]:
df_listings['has_availability'].value_counts(dropna=False)

has_availability
t      1053
NaN       2
Name: count, dtype: int64

In [82]:
cal_status_bool = df_calendar.groupby('listing_id')['available'].apply(lambda x: (x == 1).any())

In [ ]:
cal_status_map = cal_status_bool.map({True: 't', False: 'f'})

In [ ]:
df_listings['has_availability'] = df_listings['has_availability'].fillna(df_listings['id'].map(cal_status_map))

In [85]:
df_listings['has_availability'].value_counts(dropna=False)

has_availability
t    1339
f      19
Name: count, dtype: int64

In [86]:
df_listings['has_availability'].isnull().sum()

np.int64(0)

In [87]:
df_listings['has_availability'] = df_listings['has_availability'].map({'t': 1, 'f': 0}) 

In [88]:
df_listings['has_availability'].head(10)

0    1
1    1
2    1
3    1
4    1
5    1
6    1
7    1
8    1
9    1
Name: has_availability, dtype: int64

In [92]:
# 1. Tính Tỷ lệ lấp đầy cho 30 ngày (%)
# Công thức: ((Tổng số ngày - Số ngày trống) / Tổng số ngày) * 100
df_listings['occ_rate_30'] = ((30 - df_listings['availability_30']) / 30) * 100

# 2. Tính Tỷ lệ lấp đầy cho 365 ngày (%)
df_listings['occ_rate_365'] = ((365 - df_listings['availability_365']) / 365) * 100

In [93]:
df_listings['occ_rate_30'] = df_listings['occ_rate_30'].clip(lower=0, upper=100)
df_listings['occ_rate_365'] = df_listings['occ_rate_365'].clip(lower=0, upper=100)

In [94]:
df_listings['occ_rate_30'] = df_listings['occ_rate_30'].fillna(0)
df_listings['occ_rate_365'] = df_listings['occ_rate_365'].fillna(0)

In [95]:
df_listings[['occ_rate_30', 'occ_rate_365']].isnull().sum()

occ_rate_30     0
occ_rate_365    0
dtype: int64

In [101]:
mode_bath = df_listings.groupby('room_type')['bathrooms_text'].transform(
    lambda x: x.mode()[0] if not x.mode().empty else "1 bath"
)

In [ ]:
df_listings['bathrooms_text'] = df_listings['bathrooms_text'].fillna(mode_bath)

In [118]:
df_listings['room_type'].isnull().sum() 

np.int64(0)

In [117]:
df_listings['room_type'].value_counts()

room_type
Entire home/apt    593
Private room       436
Hotel room          26
Name: count, dtype: int64

In [120]:
# mã hóa biến categorical 'room_type' theo dummy encoding
df_listings = pd.get_dummies(df_listings, columns=['room_type'], drop_first=True, dtype=int)

In [121]:
room_type_cols = [col for col in df_listings.columns if 'room_type' in col]
print(f"Các cột room_type mới được tạo: {room_type_cols}")


Các cột room_type mới được tạo: ['room_type_Hotel room', 'room_type_Private room']


In [160]:
df_listings[room_type_cols].head()

,room_type_Hotel room,room_type_Private room
0,0,0
1,0,1
2,0,1
3,0,1
4,0,0


In [ ]:
# 1. Làm sạch chuỗi và chuyển thành List các tiện nghi
cleaned_amenities = df_listings['amenities'].str.replace(r'["\[\]]', '', regex=True).str.split(',')
cleaned_amenities = cleaned_amenities.apply(lambda x: [i.strip() for i in x] if isinstance(x, list) else [])

In [ ]:
# 2. Tạo cột Đếm tổng số lượng tiện nghi (amenities_count)
df_listings['amenities_count'] = cleaned_amenities.apply(len)

In [128]:
all_amenities = cleaned_amenities.explode().dropna()

In [132]:
all_amenities = all_amenities[all_amenities != ''] 


In [ ]:
# 3. Tìm các tiện nghi phổ biến nhất (Dùng hàm explode thay cho vòng lặp)

from collections import Counter
amenity_counts=Counter(all_amenities)
df_amenity_freq=pd.DataFrame(amenity_counts.items(), columns=['Amenity', 'Count']).sort_values(by='Count', ascending=False)

In [136]:
top_amenities = df_amenity_freq.head(10)['Amenity'].tolist()
print(top_amenities)

['Smoke alarm', 'Carbon monoxide alarm', 'Wifi', 'Hot water', 'Hangers', 'Essentials', 'Bed linens', 'Kitchen', 'Hair dryer', 'Refrigerator']


In [137]:
# 4. Tạo các cột nhị phân (0/1) cho Top Amenities
for amenity in top_amenities:
    # Xử lý tên cột cho đẹp: viết thường, thay khoảng trắng và dấu '/' bằng '_'
    col_name = f"has_{amenity.lower().replace(' ', '_').replace('/', '_')}"
    
    # regex=False giúp tránh lỗi nếu tên amenity có ký tự đặc biệt
    df_listings[col_name] = df_listings['amenities'].str.contains(amenity, case=False, na=False, regex=False).astype(int)


In [ ]:
df_listings[['bedrooms','beds']].isnull().sum()

In [147]:
mask = df_listings[['bedrooms', 'beds']].isnull().any(axis=1)
missing_rows = df_listings[mask]

In [151]:
print(f"{len(missing_rows)} căn hộ bị thiếu thông tin giường ngủ!")
missing_rows[['id','accommodates','price', 'bedrooms', 'beds']].head(10)

3 căn hộ bị thiếu thông tin giường ngủ!


,id,accommodates,price,bedrooms,beds
387,42116124,2,130.0,1.0,NaN
393,44473794,1,79.0,1.0,NaN
1238,1379008406971631030,2,364.0,1.0,NaN


In [ ]:
# Điền giá trị cho cột 'beds' dựa trên trung vị (median) của nhóm có cùng số phòng ngủ (bedrooms)
df_listings['beds'] = df_listings['beds'].fillna(
    df_listings.groupby('bedrooms')['beds'].transform('median')
)

In [154]:
print(df_listings.loc[[387, 393, 1238], ['id', 'accommodates', 'bedrooms', 'beds']])

                       id  accommodates  bedrooms  beds
387              42116124             2       1.0   1.0
393              44473794             1       1.0   1.0
1238  1379008406971631030             2       1.0   1.0


Xử lý missing value dựa trên suy luận logic. Trong quá trình rà soát, nhóm phát hiện 3 bản ghi bị thiếu dữ liệu tại trường 'beds''. Thông qua việc đối chiếu chéo (Cross-validation) với các trường thông tin vật lý khác, nhóm nhận thấy cả 3 căn hộ đều có 1 phòng ngủ (bedrooms = 1) và sức chứa tối đa từ 1 đến 2 người (accommodates $\le 2$).Dựa trên logic vận hành thực tế của thị trường Airbnb, nhóm áp dụng phương pháp xử lý missing value theo nhóm. Cụ thể, số lượng giường thiếu được suy từ giá trị trung vị của phân nhóm căn hộ 1 phòng ngủ (kết quả trả về là 1.0). Kỹ thuật này đảm bảo độ chính xác cao hơn so với việc xử lý missing value trung vị trên toàn bộ tập dữ liệu (Global Median).

In [301]:
cols_drop = [
    'scrape_id', 'last_scraped', 'source', 'calendar_last_scraped', 'listing_url', 'host_url', 'picture_url', 
    'host_name', 'host_id', 'neighbourhood_group_cleansed', 'calendar_updated',
    'name', 'description', 'neighborhood_overview', 'host_about', 'host_location', 
    'host_verifications', 'host_neighbourhood', 'license', 'bathrooms_text', 'bathrooms', 'amenities', 'host_thumbnail_url', 'host_picture_url'
]

df_listings.drop(columns=cols_drop, inplace=True, errors='ignore')

In [302]:
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1055 entries, 0 to 1357
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1055 non-null   int64  
 1   host_since                                    1055 non-null   object 
 2   host_response_time                            1015 non-null   object 
 3   host_response_rate                            1015 non-null   object 
 4   host_acceptance_rate                          1023 non-null   object 
 5   host_is_superhost                             1020 non-null   float64
 6   host_listings_count                           1055 non-null   int64  
 7   host_total_listings_count                     1055 non-null   int64  
 8   host_has_profile_pic                          1055 non-null   object 
 9   host_identity_verified                        1055 non-null   object

In [303]:
df_airbnb_cambridge = pd.merge(df_listings, df_occupancy, left_on='id', right_on='listing_id', how='inner')

In [304]:
df_airbnb_cambridge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1055 entries, 0 to 1054
Data columns (total 81 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1055 non-null   int64  
 1   host_since                                    1055 non-null   object 
 2   host_response_time                            1015 non-null   object 
 3   host_response_rate                            1015 non-null   object 
 4   host_acceptance_rate                          1023 non-null   object 
 5   host_is_superhost                             1020 non-null   float64
 6   host_listings_count                           1055 non-null   int64  
 7   host_total_listings_count                     1055 non-null   int64  
 8   host_has_profile_pic                          1055 non-null   object 
 9   host_identity_verified                        1055 non-null   o

In [305]:
df_airbnb_cambridge.isnull().sum()

id                            0
host_since                    0
host_response_time           40
host_response_rate           40
host_acceptance_rate         32
                             ..
has_dishes_and_silverware     0
has_tv                        0
verifications_count           0
listing_id                    0
occupancy_rate                0
Length: 81, dtype: int64

In [306]:
cols = ['beds', 'bedrooms', 'bathrooms_count', 'host_response_rate', 'host_acceptance_rate']
for col in cols:
    if df_airbnb_cambridge[col].dtype == 'object':
        df_airbnb_cambridge[col] = df_airbnb_cambridge[col].str.replace('%', '').astype(float)
    df_airbnb_cambridge[col] = df_airbnb_cambridge[col].fillna(df_airbnb_cambridge[col].median())

# Điền 'Unknown' cho các biến phân loại
df_airbnb_cambridge['host_response_time'] = df_airbnb_cambridge['host_response_time'].fillna('Unknown')
df_airbnb_cambridge['host_is_superhost'] = df_airbnb_cambridge['host_is_superhost'].fillna('Unknown')

In [307]:
df_airbnb_cambridge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1055 entries, 0 to 1054
Data columns (total 81 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            1055 non-null   int64  
 1   host_since                                    1055 non-null   object 
 2   host_response_time                            1055 non-null   object 
 3   host_response_rate                            1055 non-null   float64
 4   host_acceptance_rate                          1055 non-null   float64
 5   host_is_superhost                             1055 non-null   object 
 6   host_listings_count                           1055 non-null   int64  
 7   host_total_listings_count                     1055 non-null   int64  
 8   host_has_profile_pic                          1055 non-null   object 
 9   host_identity_verified                        1055 non-null   o

In [310]:
df_airbnb_cambridge['host_has_profile_pic']

0       t
1       t
2       t
3       t
4       t
       ..
1050    t
1051    t
1052    t
1053    t
1054    t
Name: host_has_profile_pic, Length: 1055, dtype: object

In [311]:
df_airbnb_cambridge['host_identity_verified']

0       t
1       t
2       t
3       t
4       t
       ..
1050    t
1051    t
1052    t
1053    t
1054    t
Name: host_identity_verified, Length: 1055, dtype: object

In [308]:
df_airbnb_cambridge.head(10)


,id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_has_profile_pic,host_identity_verified,...,has_self_check-in,has_dedicated_workspace,has_heating,has_cooking_basics,has_air_conditioning,has_dishes_and_silverware,has_tv,verifications_count,listing_id,occupancy_rate
0,8521,2010-12-01,within a few hours,100.0,65.0,1.0,2,3,t,t,...,0,1,1,1,1,1,1,2,8521,0.257534
1,11169,2009-09-24,within a day,100.0,77.0,1.0,4,4,t,t,...,0,1,1,0,1,1,0,2,11169,0.852055
2,19581,2010-01-27,within an hour,100.0,100.0,1.0,3,3,t,t,...,1,1,1,1,0,1,1,2,19581,0.805479
3,27498,2010-01-27,within an hour,100.0,100.0,1.0,3,3,t,t,...,1,1,1,1,0,1,1,2,27498,0.934247
4,79762,2011-03-08,within a few hours,100.0,89.0,1.0,1,2,t,t,...,1,0,1,1,1,1,1,2,79762,0.824658
5,106474,2010-01-27,within an hour,100.0,100.0,1.0,3,3,t,t,...,1,1,1,1,0,1,1,2,106474,0.956164
6,108898,2010-12-01,within a few hours,100.0,65.0,1.0,2,3,t,t,...,0,1,1,1,0,1,1,2,108898,0.046575
7,675441,2012-08-30,Unknown,100.0,96.0,0.0,2,5,t,t,...,0,0,1,0,1,0,1,2,675441,1.000000
8,715532,2012-09-25,within a day,100.0,97.0,1.0,1,3,t,t,...,1,0,1,1,0,1,1,3,715532,0.657534
9,742574,2011-09-13,a few days or more,0.0,0.0,0.0,5,6,t,f,...,1,1,1,1,1,1,1,2,742574,0.326027
